# Regressões MODIS → VIIRS por país

Compara modelos Quasi-Poisson e NB2 para as médias exponencial, potencial e proporcional na Argentina (AQUA → SNPP). Após a comparação, permite escolher explicitamente uma família e uma hipótese para reestimar os coeficientes em cada país/par.

A avaliação é no próprio ajuste (in-sample); RMSE/MAE não representam desempenho futuro. Usa somente meses presentes para ambos os satélites no mesmo país, sem preencher meses ausentes. Janeiro/2012 permanece porque consta no dataset. A etapa global publica os modelos, as previsões mensais e os pesos espaciais no DuckDB e nos CSVs.

As funções de carregamento, ajuste, gráficos e persistência estão em `dataset_processing/regression/regression_pipeline.py`. Instale as dependências com `pip install -r dataset_processing/regression/requirements.txt`.

## 1. Imports e configuração do projeto

In [ ]:
from pathlib import Path
import importlib
import pandas as pd
import sys
import warnings
from IPython.display import display

CWD = Path.cwd().resolve()
PROJECT_ROOT = next((path for path in (CWD, *CWD.parents) if (path / 'dataset_processing' / 'regression').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Não foi possível localizar dataset_processing/regression a partir do diretório atual.')
REGRESSION_DIR = PROJECT_ROOT / 'dataset_processing' / 'regression'
if str(REGRESSION_DIR) not in sys.path:
    sys.path.insert(0, str(REGRESSION_DIR))
import regression_pipeline
importlib.reload(regression_pipeline)  # recarrega a API/esquema atuais mesmo se o kernel já importou uma versão anterior
from regression_pipeline import (
    FAMILIES, HYPOTHESES, PAIRS, EXPECTED_SENSOR, Fit,
    calculate_metrics, comparison_ranking, coverage_summary, fit_model,
    load_monthly_counts, make_record, model_equation, plot_diagnostic,
    prepare_pair, predict_equation, save_selected_models,
)

CAMINHO_BANCO = (PROJECT_ROOT / 'data' / 'focos_america_sul.duckdb').resolve()
PAISES = None  # None = todos os nomes distintos no banco
PARES = list(PAIRS)
PERIODO_INICIAL = None  # YYYY-MM-01, inclusivo
PERIODO_FINAL = None  # YYYY-MM-01, inclusivo
SEMENTE = 20261112
FAMILIA_ESCOLHIDA = 'quasi_poisson'  # escolha global confirmada [quasi_poisson/nb2]
HIPOTESE_ESCOLHIDA = 'potencial'  # escolha global confirmada
EXIBIR_GRAFICOS_GLOBAIS = False
PUBLICAR_RESULTADOS = False  # altere explicitamente para True após revisar os ajustes
publication_succeeded = False

if not CAMINHO_BANCO.is_file():
    raise FileNotFoundError(f'Banco DuckDB não encontrado: {CAMINHO_BANCO}')
print(f'Banco: {CAMINHO_BANCO}')

## 2. Carregar e validar as contagens mensais

In [ ]:
base = load_monthly_counts(CAMINHO_BANCO)
zero_rows = int((base['nfocos'] == 0).sum())
print(f'Linhas mensais: {len(base):,}; países/territórios: {base.pais.nunique()}; contagens zero: {zero_rows:,}')
display(coverage_summary(base))
pais_disponiveis = sorted(base['pais'].unique().tolist())
if PAISES is None:
    PAISES_ETAPA_B = pais_disponiveis
else:
    desconhecidos = sorted(set(PAISES) - set(pais_disponiveis))
    if desconhecidos:
        raise ValueError(f'Países não encontrados no banco: {desconhecidos}')
    PAISES_ETAPA_B = list(PAISES)
print('Países selecionados para a etapa global:', len(PAISES_ETAPA_B))

## 3. Comparação — Argentina / AQUA → SNPP
Os seis candidatos usam exatamente a mesma interseção de meses. Erros são medidos na escala original da resposta e no próprio ajuste.

In [ ]:
argentina = prepare_pair(base, 'Argentina', 'AQUA', 'SNPP', PERIODO_INICIAL, PERIODO_FINAL)
if argentina.frame.empty:
    raise RuntimeError('Argentina / AQUA → SNPP não tem meses pareados no período configurado.')
print(f"Meses pareados: {len(argentina.frame)}; período: {argentina.frame.data.min():%Y-%m} a {argentina.frame.data.max():%Y-%m}; meses ausentes na janela: {argentina.absent_months}")
argentina_zero = int(((argentina.frame.x == 0) | (argentina.frame.y == 0)).sum())
print(f'Meses pareados com zero em x ou y: {argentina_zero}')
if argentina_zero:
    print('Os zeros serão preservados. Apenas o candidato proporcional é incompatível se x=0.')

ajustes_argentina = {}
registros_argentina = []
for family in FAMILIES:
    for hypothesis in HYPOTHESES:
        fit = fit_model(argentina.frame, family, hypothesis, SEMENTE)
        metrics, diag = calculate_metrics(argentina.frame, fit) if fit.status == 'ok' else ({}, None)
        ajustes_argentina[(family, hypothesis)] = {'fit': fit, 'metrics': metrics, 'diagnostics': diag}
        record = make_record('Argentina', 'AQUA', 'SNPP', family, hypothesis, argentina, fit, diag)
        registros_argentina.append(record)

ranking_columns = ['familia', 'hipotese', 'status', 'rmse', 'mae', 'vies_medio', 'maior_erro_absoluto', 'meses_maior_erro', 'motivo_status']
ranking = comparison_ranking(registros_argentina)
display((ranking if not ranking.empty else pd.DataFrame(registros_argentina))[ranking_columns])
if not ranking.empty:
    melhor = ranking.iloc[0]
    print(f"Melhor ajuste in-sample na Argentina pelo MAE: {melhor.familia} / {melhor.hipotese}. Isso não estima desempenho futuro.")
else:
    print('Nenhum candidato argentino teve ajuste válido; confira os motivos de falha acima.')
print('Os seis candidatos da comparação permanecem somente em memória e não entram em modelos_equacoes.')

## 4. Três diagnósticos da comparação
Cada diagnóstico inclui os seis candidatos em uma grade de três hipóteses por duas famílias. Em candidatos inválidos, o painel informa o motivo. Ao final, o modelo escolhido na configuração é comparado com os valores SNPP observados, mês a mês, no período de ajuste da Argentina.

In [ ]:
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm

def plot_comparison(kind):
    fig, axes = plt.subplots(3, 2, figsize=(14, 15), squeeze=False)
    for row, hypothesis in enumerate(HYPOTHESES):
        for col, family in enumerate(FAMILIES):
            item = ajustes_argentina[(family, hypothesis)]
            fit = item['fit']
            ax = axes[row, col]
            if fit.status != 'ok':
                ax.text(0.5, 0.5, fit.reason or fit.status, ha='center', va='center', wrap=True)
                ax.set_title(f'{family} | {hypothesis} | {fit.status}')
                ax.axis('off')
                continue
            diag = item['diagnostics'].sort_values('data')
            x = diag.x.to_numpy(dtype=float)
            y = diag.y.to_numpy(dtype=float)
            mu = diag.media_prevista.to_numpy(dtype=float)
            if kind == 'curva média':
                ax.scatter(x, y, s=22, alpha=0.65, label='Observado')
                grid = np.linspace(float(x.min()), float(x.max()), 400)
                ax.plot(grid, predict_equation(fit, grid), color='darkorange', lw=2, label='Média estimada')
                ax.set_xlabel('Contagem mensal AQUA')
                ax.set_ylabel('Contagem mensal SNPP')
                ax.legend()
            elif kind == 'observado × previsto':
                bounds = [min(float(mu.min()), float(y.min())), max(float(mu.max()), float(y.max()))]
                ax.scatter(mu, y, s=22, alpha=0.65)
                ax.plot(bounds, bounds, 'k--', lw=1, label='1:1')
                ax.set_xlim(bounds)
                ax.set_ylim(bounds)
                ax.set_aspect('equal', adjustable='box')
                ax.set_xlabel('Média prevista')
                ax.set_ylabel('Observado')
                ax.legend()
                ax.text(0.02, 0.98, f"RMSE={item['metrics']['rmse']:.4g}; MAE={item['metrics']['mae']:.4g}", transform=ax.transAxes, va='top')
            else:
                ax.axhline(0, color='black', lw=1)
                ax.scatter(diag.data, diag.residuo_bruto, s=22, alpha=0.7)
                ax.set_xlabel('Mês')
                ax.set_ylabel('Observado − previsto')
            ax.set_title(f'{family} | {hypothesis}')
            ax.grid(alpha=0.2)
    fig.suptitle(f'Argentina | AQUA → SNPP | {kind} | avaliação in-sample', y=1.01)
    fig.tight_layout()
    return fig

figuras_comparacao = {}
for tipo_grafico in ('curva média', 'observado × previsto', 'resíduos temporais'):
    fig = plot_comparison(tipo_grafico)
    figuras_comparacao[tipo_grafico] = fig
    display(fig)
    plt.close(fig)

chave_escolhida = (FAMILIA_ESCOLHIDA, HIPOTESE_ESCOLHIDA)
if FAMILIA_ESCOLHIDA is None or HIPOTESE_ESCOLHIDA is None:
    print('Defina FAMILIA_ESCOLHIDA e HIPOTESE_ESCOLHIDA na configuração para plotar o modelo selecionado.')
elif chave_escolhida not in ajustes_argentina:
    raise ValueError(f'Modelo argentino escolhido não encontrado: {chave_escolhida}')
else:
    ajuste_escolhido = ajustes_argentina[chave_escolhida]
    if ajuste_escolhido['fit'].status != 'ok' or ajuste_escolhido['diagnostics'] is None:
        print(f"O modelo argentino {chave_escolhida} não gerou previsões válidas: {ajuste_escolhido['fit'].reason}")
    else:
        diag_escolhido = ajuste_escolhido['diagnostics'].sort_values('data').reset_index(drop=True)
        argentina_ordenada = argentina.frame.sort_values('data').reset_index(drop=True)
        if not diag_escolhido.data.equals(argentina_ordenada.data):
            raise RuntimeError('As datas dos diagnósticos não correspondem aos meses usados no ajuste argentino.')
        diag_escolhido['erro_absoluto'] = diag_escolhido.residuo_bruto.abs()
        pior_mes = diag_escolhido.loc[diag_escolhido.erro_absoluto.idxmax()]
        print(f"Maior erro absoluto mensal ({FAMILIA_ESCOLHIDA} / {HIPOTESE_ESCOLHIDA}): "
              f"{pior_mes.data:%Y-%m} = {pior_mes.erro_absoluto:,.2f} focos "
              f"(observado={pior_mes.y:,.0f}; estimado={pior_mes.media_prevista:,.2f}).")
        fig, ax = plt.subplots(figsize=(15, 6))
        ax.plot(diag_escolhido.data, diag_escolhido.media_prevista, color='darkorange',
                lw=2, marker='o', markersize=3, zorder=2, label='SNPP estimado pelo modelo')
        ax.plot(diag_escolhido.data, diag_escolhido.y, color='royalblue',
                lw=1.5, marker='o', markersize=3, zorder=3, label='SNPP observado')
        ax.vlines(pior_mes.data, min(pior_mes.y, pior_mes.media_prevista),
                  max(pior_mes.y, pior_mes.media_prevista), color='black', ls=':', lw=1.5,
                  label=f'Maior erro absoluto: {pior_mes.erro_absoluto:,.2f} ({pior_mes.data:%Y-%m})')
        ax.annotate(f"Erro = {pior_mes.erro_absoluto:,.2f}",
                    xy=(pior_mes.data, (pior_mes.y + pior_mes.media_prevista) / 2),
                    xytext=(10, 12), textcoords='offset points',
                    arrowprops={'arrowstyle': '->', 'color': 'black'})
        ax.set(title=f'Argentina | AQUA → SNPP | {FAMILIA_ESCOLHIDA} / {HIPOTESE_ESCOLHIDA} | ajuste in-sample',
               xlabel='Ano-mês', ylabel='Quantidade mensal de focos')
        ax.xaxis.set_major_locator(mdates.YearLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        ax.grid(alpha=0.2)
        ax.legend()
        fig.autofmt_xdate(rotation=45)
        fig.tight_layout()
        display(fig)
        plt.close(fig)

## 5. Escolha explícita e ajuste por país
Escolha família e hipótese após ver o ranking. Para publicar no DuckDB e nos CSVs, também altere `PUBLICAR_RESULTADOS` para `True`; o valor padrão `False` torna **Run All** seguro. No kernel interativo, o notebook pergunta os dois valores apenas quando ambos estão vazios.

In [ ]:
familia_global = FAMILIA_ESCOLHIDA
hipotese_global = HIPOTESE_ESCOLHIDA
try:
    from IPython import get_ipython
    kernel_interativo = get_ipython() is not None
except Exception:
    kernel_interativo = False
if familia_global is None and hipotese_global is None and kernel_interativo:
    familia_global = input('Família global [quasi_poisson/nb2] (vazio para encerrar): ').strip().lower() or None
    hipotese_global = input('Hipótese global [exponencial/potencial/proporcional] (vazio para encerrar): ').strip().lower() or None
if familia_global is None and hipotese_global is None:
    print('Comparação argentina concluída. Escolha global vazia; etapa por país não executada.')
elif familia_global not in FAMILIES or hipotese_global not in HYPOTHESES:
    raise ValueError(f'Escolha inválida: família={familia_global!r}, hipótese={hipotese_global!r}')
else:
    registros_globais = []
    ajustes_globais = {}
    for country in tqdm(PAISES_ETAPA_B, desc='Regressões por país'):
        for origin, destination in PARES:
            prepared = prepare_pair(base, country, origin, destination, PERIODO_INICIAL, PERIODO_FINAL)
            fit = fit_model(prepared.frame, familia_global, hipotese_global, SEMENTE)
            metrics, diag = calculate_metrics(prepared.frame, fit) if fit.status == 'ok' else ({}, None)
            ajustes_globais[(country, origin, destination)] = {'prepared': prepared, 'fit': fit, 'metrics': metrics, 'diagnostics': diag}
            registros_globais.append(make_record(country, origin, destination, familia_global, hipotese_global, prepared, fit, diag))
            if EXIBIR_GRAFICOS_GLOBAIS:
                period = f'{prepared.frame.data.min():%Y-%m}–{prepared.frame.data.max():%Y-%m}' if not prepared.frame.empty else 'sem período pareado'
                for kind in ('curva média', 'observado × previsto', 'resíduos temporais'):
                    display(plot_diagnostic(prepared.frame, fit, country, origin, destination, familia_global, hipotese_global, period, kind))
    falhas = [r for r in registros_globais if r['status'] != 'ok']
    if falhas:
        print(f'{len(falhas)} ajuste(s) falharam. A tabela atual foi preservada; corrija/inspecione os casos antes de substituir os resultados.')
        display(pd.DataFrame(falhas)[['pais', 'satelite_origem', 'satelite_destino', 'status', 'motivo_status']])
    elif not PUBLICAR_RESULTADOS:
        print('Ajustes globais concluídos e validados, mas não publicados. Defina PUBLICAR_RESULTADOS=True para persistir os resultados.')
    else:
        caminho_csv_mensal = PROJECT_ROOT / 'data' / 'focos_mensais_america_sul.csv'
        caminho_csv_bruto = PROJECT_ROOT / 'data' / 'focos_brutos_america_sul.csv'
        n_global = save_selected_models(CAMINHO_BANCO, registros_globais, PAISES_ETAPA_B, PARES, caminho_csv_mensal, caminho_csv_bruto)
        publication_succeeded = True
        print(f'{n_global} modelos SNPP, previsões mensais e pesos por foco salvos no DuckDB e nos CSVs.')
        import duckdb
        with duckdb.connect(str(CAMINHO_BANCO), read_only=True) as con:
            resumo_final = con.execute('SELECT pais, satelite_origem, satelite_destino, familia, hipotese, equacao, rmse, mae, vies_medio, n_meses FROM modelos_equacoes ORDER BY pais, satelite_origem, satelite_destino').fetchdf()
            resumo_harmonizacao = con.execute('SELECT count(*) FILTER (WHERE nfocos_harm IS NOT NULL), count(*) FILTER (WHERE nfocos IS NULL), coalesce(sum(nfocos), 0) FROM focos_mensais').fetchone()
        print(f'Previsões harmonizadas: {resumo_harmonizacao[0]:,}; linhas VIIRS sintéticas: {resumo_harmonizacao[1]:,}; soma observada nfocos: {resumo_harmonizacao[2]:,}.')
        display(resumo_final)

## 6. Série histórica completa — Argentina
Compara AQUA observado, SNPP observado e a previsão mensal do modelo na escala SNPP. Execute após a publicação bem-sucedida da etapa global.

In [ ]:
import duckdb
import matplotlib.dates as mdates
import matplotlib.pyplot as plt

if not globals().get('publication_succeeded', False):
    print('Gráfico histórico ignorado: esta execução não publicou novos resultados.')
else:
    with duckdb.connect(str(CAMINHO_BANCO), read_only=True) as con:
        historico_argentina = con.execute("""
        SELECT make_date(ano, mes, 1) AS data,
               max(nfocos_harm) FILTER (WHERE satelite='SNPP') AS harmonizado,
               max(nfocos) FILTER (WHERE satelite='SNPP') AS snpp_observado,
               max(nfocos) FILTER (WHERE satelite='AQUA') AS aqua_observado
        FROM focos_mensais
        WHERE pais='Argentina' AND satelite IN ('SNPP', 'AQUA')
        GROUP BY ano, mes
        ORDER BY ano, mes
        """).fetchdf()

    if historico_argentina.empty:
        raise RuntimeError('Não há dados históricos para plotar na Argentina.')
    calendario = pd.date_range(historico_argentina.data.min(), historico_argentina.data.max(), freq='MS')
    historico_argentina = (historico_argentina.set_index('data').reindex(calendario)
                          .rename_axis('data').reset_index())
    fig, ax = plt.subplots(figsize=(15, 6))
    ax.plot(historico_argentina.data, historico_argentina.aqua_observado, color='seagreen',
            lw=1.0, alpha=0.75, zorder=1, label='AQUA observado')
    ax.plot(historico_argentina.data, historico_argentina.harmonizado, color='darkorange', lw=2, zorder=2,
            label='Previsão harmonizada (escala SNPP)')
    ax.plot(historico_argentina.data, historico_argentina.snpp_observado, color='royalblue',
            lw=1.2, alpha=1.0, marker='o', markersize=2.8, markevery=6, zorder=4,
            label='SNPP observado')
    ax.axvline(pd.Timestamp('2012-01-01'), color='gray', ls='--', lw=1,
               label='Início do período SNPP observado')
    ax.set(title='Argentina | série histórica mensal de focos',
           xlabel='Ano-mês', ylabel='Quantidade mensal de focos')
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.grid(alpha=0.2)
    ax.legend()
    fig.autofmt_xdate(rotation=45)
    fig.tight_layout()
    display(fig)
    plt.close(fig)